In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# 某电商平台 2024 上半年真实销售数据（含缺失值和脏数据）
n = 200
df = pd.DataFrame({
    '订单号': [f'ORD{20240000 + i}' for i in range(1, n + 1)],
    '日期': pd.date_range('2024-01-01', periods=n, freq='10h'),  # ← 注意这里是小写 h
    '销售员': np.random.choice(['小王', '小李', '小张', '小陈', '小赵'], n),
    '地区': np.random.choice(['华东', '华南', '华北', '华西', np.nan], n, p=[0.3, 0.25, 0.25, 0.15, 0.05]),
    '商品类别': np.random.choice(['电子', '服装', '食品', '日用品', '图书'], n),
    '单价': np.random.choice([89, 199, 299, 599, 1299, 2499, 4999], n),
    '数量': np.random.choice([1, 1, 1, 2, 2, 3, 5, 10, np.nan], n)
})

# 计算销售额
df['销售额'] = df['单价'] * df['数量']

# 故意加一些脏数据
df.loc[[10, 50, 120], '销售额'] = np.nan       # 缺失值
df.loc[[5, 30], '数量'] = -5                    # 负数（异常值）
df.loc[[80, 81], '销售额'] = 99999              # 离谱的值
df.loc[145, '销售员'] = np.nan                  # 缺销售员


#查看数量的中位值
num_median = df['数量'].median()
print(num_median)
# 填充缺失值
df = df.fillna({
    '销售员': '未知销售员',
    '数量': df['数量'].median(),
    '销售额': df['单价'] * 2.0,
    '地区': '未知地区'

})

# 修改错误数据
df.loc[df['数量'] < 0, '数量'] = df['数量'].median()
df.loc[df['销售额'] == 99999, '销售额'] = df.loc[df['销售额'] == 99999, '单价'] * df.loc[df['销售额'] == 99999, '数量']
#查看缺失值
df.isnull().sum()

In [ ]:
#统计最大总销售额的值和人
person_sales = df.groupby('销售员')['销售额'].sum()
print("销售额最大的销售员：", person_sales.idxmax(), "销售额：", person_sales.max())
#统计哪个地区的总销售额最大
location_sales = df.groupby('地区')['销售额'].sum()
print("总销售额最大的地区：", location_sales.idxmax(), "销售额：", location_sales.max())
#统计哪个商品类别的总销量最大
best_product =df.groupby('商品类别')['数量'].sum()
print('销量最好的商品为：',best_product.idxmax(),'销量为：',int(best_product.max()))
#哪个月销售额最大
#创建月份列
df['月份'] = df['日期'].dt.month
month_sales = df.groupby('月份')['销售额'].sum()
print('每个月的销售额为：')
print(month_sales)
print('销售额最大的月份为：',month_sales.idxmax(),'销售额为：',month_sales.max())

#统计人均客单价
person_mean_sales = df['销售额'].sum() / df['数量'].sum()
print('人均客单价为：',round(person_mean_sales, 2))
#统计退货数量最多的地区 但是没有数据
# back_product = df.groupby('地区')['退货数量'].sum()
# print('退货数量最多的地区为：',back_product.idxmax(),'退货数量为：',back_product.max())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

#设置中文
sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

#绘制图，都用seaborn绘制
#设置画板
plt.figure(figsize=(12,8))
#第一个子图，总6个
#设置数据
sum_sales = df.groupby('销售员')['销售额'].sum()
plt.subplot(2,3,1)
sns.barplot(x=sum_sales.index, y=sum_sales.values,palette='Blues_r')
#显示顶部数据
for a,b in zip(sum_sales.index,sum_sales.values):
    plt.text(a,b+300,b,ha='center',va='bottom',fontsize=10)
plt.title('每个销售员的总销售额')
plt.ylabel('销售额')

#第二个子图
plt.subplot(2,3,2)
#设置数据
df ['月份'] = df['日期'].dt.month
month_sales = df.groupby('月份')['销售额'].sum()
#绘制折线图
sns.lineplot(x=month_sales.index, y=month_sales.values, marker='o', color='orange', linewidth=2.5, markersize=8, label='销售额')
#显示顶部数据
for a,b in zip(month_sales.index,month_sales.values):
    plt.text(a,b+300,b,ha='center',va='bottom',fontsize=10)
plt.title('每月销售额')
plt.ylabel('销售额')

#第三个子图
plt.subplot(2,3,3)
#设置数据 
#将地区为空的数据设置为'其他'
df.loc[df['地区'].isnull(), '地区'] = '其他'
location_sales = df.groupby('地区')['销售额'].sum()

#绘制饼图
plt.pie(location_sales.values, labels=location_sales.index, autopct='%1.1f%%', startangle=90, colors=['#FFA500', '#FFC0CB', '#ADD8E6', '#90EE90', '#D3D3D3'])

plt.legend(loc='upper right')
plt.title('各地区销售额分布')
plt.axis('equal')

#第四个子图
plt.subplot(2,3,4)
#设置数据
product_category = df.groupby('商品类别')['单价']
#绘制箱线图
sns.boxplot(x='商品类别', y='单价', data=df, palette='Set3')
plt.title('商品类别单价箱线图')
plt.ylabel('单价')
plt.xlabel('商品类别')


#第五个子图
plt.subplot(2,3,5)
#设置数据
corr = df[['单价','数量','销售额']].corr()
#绘制热力图
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('数值列相关性热力图', fontsize=13)

#第六个子图
plt.subplot(2,3,6)
#设置数据
#绘制散点图
sns.scatterplot(x='单价', y='销售额', data=df, hue='销售员', palette='Set1')
plt.title('单价与销售额的散点图')




plt.tight_layout()
plt.show()

In [1]:
print('数据清洗和分析完成！')

数据清洗和分析完成！


# 电商销售数据分析报告

## 一、数据概况
本次分析基于 2024 年上半年 200 条电商销售订单，涵盖 5 名销售员、4 个地区、5 类商品。

## 二、核心发现
小李的销售业绩表现差劲，贡献全平台 73499 销售额
2 月整体销售额达到全年峰值 317051，1 月则有 308524，但 3 月销售额直接暴跌至 214272，说明销售额不稳定
华北地区贡献全平台 33.4% 销售额，是核心营收区域
商品类别单价箱线图可见，全部品类都存在极高单价离群点，可能存在脏数据或异常值
销售额和单价相关系数 0.58、销售额和购买数量相关系数 0.55；单价和数量几乎无关联（-0.01），说明客单价、采购量都会同步带动销售额提升，二者互不冲突。

### 1. 销售冠军
小王以 231,276 元总销售额位居第一...

### 2. 地区分布
华北区占比 33.4%...

...

## 三、建议
- 针对 3 月销售额下滑，建议增加促销活动
- 食品类异常单价订单需排查是否为录入错误
培训小李
优化数据质量，对异常单价订单进行排查和清理
